# Lab 18-01 · Memory Maps, Wait States and Clock Trees

Six parts, each built on `mcuarch.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import mcuarch as ma
rng = np.random.default_rng(1801)
T_ACC = 33.3e-9  # flash access time of Example 18-01.2

## Part 1 · Addresses

*Predict:* the bit-band alias of bit 13 of the word at 0x4001 1008, and the byte and bit it really changes.

In [2]:
addr, bit = 0x40011008, 13
alias = ma.bitband_alias(addr, bit)
print(f"region of 0x{addr:08X}: {ma.region(addr)}")
print(f"alias of bit {bit}: 0x{alias:08X}")
# decode it back: alias = base_alias + 32 * byte_offset + 4 * bit_in_byte
base_alias, base = (0x42000000, 0x40000000) if alias >= 0x42000000 else (0x22000000, 0x20000000)
off = alias - base_alias
byte, bit_in_byte = base + off // 32, (off % 32) // 4
print(f"decodes to byte 0x{byte:08X}, bit {bit_in_byte} (word 0x{byte & ~3:08X}, bit {8 * (byte & 3) + bit_in_byte})")
for a in (0x08001234, 0x20008000, 0x40013800, 0xE000E100, 0x60000000):
    print(f"0x{a:08X}: {ma.region(a)}")

region of 0x40011008: Peripheral
alias of bit 13: 0x42220134
decodes to byte 0x40011009, bit 5 (word 0x40011008, bit 13)
0x08001234: Code
0x20008000: SRAM
0x40013800: Peripheral
0xE000E100: Private peripheral bus
0x60000000: External RAM


## Part 2 · Wait states

*Predict:* the instruction rate at 30 MHz and at 31 MHz (16-bit code, 0.12 taken branches per instruction, prefetch only, 33.3 ns flash).

In [3]:
def rate(f, b=0.12):
    ws = ma.wait_states(f, T_ACC)
    return f / 1e6 / ma.cpi_analytic(ws, 16, 2, b)

for f in (30e6, 31e6):
    print(f"{f / 1e6:.0f} MHz: {ma.wait_states(f, T_ACC)} wait states, {rate(f):.1f} instructions per us")
fs = np.arange(1e6, 180e6 + 1, 0.25e6)
plt.plot(fs / 1e6, [rate(f) for f in fs], label="prefetch only")
plt.plot(fs / 1e6, fs / 1e6, "--", label="zero wait states")
plt.xlabel("clock (MHz)"); plt.ylabel("instructions per us"); plt.legend(); plt.show()
tops = [ma.max_freq(w, T_ACC) / 1e6 for w in range(6)]
print("top of each band (MHz):", [f"{t:.1f}" for t in tops])

30 MHz: 0 wait states, 30.0 instructions per us
31 MHz: 1 wait states, 27.7 instructions per us
top of each band (MHz): ['30.0', '60.1', '90.1', '120.1', '150.2', '180.2']


## Part 3 · The fetch simulator

*Predict:* for aligned 16-bit code, does the simulator match `cpi_analytic`? What do unaligned targets and 32-bit instructions add?

In [4]:
aligned = ma.make_program(60_000, 5, 0.6, 0.0, rng, aligned=True)
mixed = ma.make_program(60_000, 5, 0.6, 0.3, rng, aligned=False)
print(" WS   aligned-sim  formula   mixed/unaligned-sim")
for ws in range(8):
    a = ma.simulate_fetch(aligned, ws).cpi
    m = ma.simulate_fetch(mixed, ws).cpi
    taken = sum(t for _, _, t in aligned) / len(aligned)
    print(f"{ws:3d}   {a:9.3f}   {ma.cpi_analytic(ws, 16, 2, taken):7.3f}   {m:9.3f}")

 WS   aligned-sim  formula   mixed/unaligned-sim
  0       1.000     1.000       1.005


  1       1.120     1.120       1.149


  2       1.240     1.240       1.312
  3       1.360     1.360       1.494
  4       1.480     1.480       1.697


  5       1.600     1.600       1.918
  6       1.719     1.720       2.169


  7       1.839     1.840       2.434


## Part 4 · Caches

*Predict:* which program, a loop in 512 bytes or random branching over 64 KiB, gains from a 16-line and a 64-line cache at 5 wait states?

In [5]:
loop = ma.make_program(60_000, 5, 0.6, 0.0, rng, aligned=True, code_bytes=512)
rand = ma.make_program(60_000, 5, 0.6, 0.0, rng, aligned=True, code_bytes=64 * 1024)
for name, prog in (("loop, 512 B", loop), ("random, 64 KiB", rand)):
    print(name, [f"{ma.simulate_fetch(prog, 5, cache_lines=c).cpi:.3f}" for c in (0, 16, 64)], "(0, 16, 64 lines)")

loop, 512 B ['1.570', '1.299', '1.002'] (0, 16, 64 lines)


random, 64 KiB ['1.595', '1.593', '1.587'] (0, 16, 64 lines)


## Part 5 · PLL

*Predict:* is an exact 48 MHz USB clock possible together with 168, 120 and 180 MHz from an 8 MHz crystal?

In [6]:
for f in (168e6, 120e6, 180e6):
    free = ma.pll_search(8e6, f)
    with48 = ma.pll_search(8e6, f, f_aux=48e6)
    print(f"{f / 1e6:.0f} MHz: {len(free)} configurations, {len(with48)} with exact 48 MHz")
    for m, n, p, q, fo, fa in with48:
        print(f"    M={m:2d} N={n:3d} P={p} Q={q:2d}: VCO {8e6 / m * n / 1e6:.0f} MHz")

168 MHz: 5 configurations, 5 with exact 48 MHz
    M= 4 N=168 P=2 Q= 7: VCO 336 MHz
    M= 5 N=210 P=2 Q= 7: VCO 336 MHz
    M= 6 N=252 P=2 Q= 7: VCO 336 MHz
    M= 7 N=294 P=2 Q= 7: VCO 336 MHz
    M= 8 N=336 P=2 Q= 7: VCO 336 MHz
120 MHz: 5 configurations, 5 with exact 48 MHz
    M= 4 N=120 P=2 Q= 5: VCO 240 MHz
    M= 5 N=150 P=2 Q= 5: VCO 240 MHz
    M= 6 N=180 P=2 Q= 5: VCO 240 MHz
    M= 7 N=210 P=2 Q= 5: VCO 240 MHz
    M= 8 N=240 P=2 Q= 5: VCO 240 MHz
180 MHz: 5 configurations, 0 with exact 48 MHz


## Part 6 · Energy

*Predict:* with a fixed current of 0.2 mA and of 2 mA, is the best clock lower or higher than Example 18-01.5's 60 MHz?

In [7]:
def energy(f, i0, n=1.2e6, period=0.1, vdd=1.8, k=40e-12, isl=3e-6):
    cyc = n * ma.cpi_analytic(ma.wait_states(f, T_ACC), 16, 2, 0.12)
    return ma.duty_cycle_energy(f, cyc, period, vdd, i0, k, isl)

fs = np.arange(1e6, 168e6 + 1, 0.25e6)
for i0 in (0.2e-3, 0.6e-3, 2e-3):
    e = np.array([energy(f, i0) for f in fs])
    best = fs[np.argmin(e)]
    plt.plot(fs / 1e6, e * 1e6, label=f"I0 = {i0 * 1e3:.1f} mA")
    print(f"I0 = {i0 * 1e3:.1f} mA: best clock {best / 1e6:.0f} MHz, {e.min() * 1e6:.1f} uJ per period")
plt.xlabel("clock (MHz)"); plt.ylabel("energy per period (uJ)"); plt.legend(); plt.show()

I0 = 0.2 mA: best clock 30 MHz, 101.1 uJ per period
I0 = 0.6 mA: best clock 60 MHz, 121.4 uJ per period
I0 = 2.0 mA: best clock 120 MHz, 166.9 uJ per period


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).